# Problem 2 — Linear Regression in Amazon SageMaker

This notebook demonstrates two approaches:

1. Without a user-created container: train directly in the SageMaker notebook kernel and save/upload the artifact.
2. With container technology: build a custom Docker training image, push it to Amazon ECR, and start a SageMaker training job.

In [ ]:
import os
import boto3
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

data = pd.read_csv("winequality-red.csv", sep=";")
X = data.drop(columns=["quality"])
y = data["quality"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

## Part A — SageMaker notebook workflow without a user-created container

In [ ]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", LinearRegression())
])
model.fit(X_train, y_train)
pred = model.predict(X_test)

metrics_a = {
    "MAE": mean_absolute_error(y_test, pred),
    "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
    "R2": r2_score(y_test, pred)
}
metrics_a

In [ ]:
joblib.dump(model, "linear_regression_no_custom_container.pkl")
print("Model artifact saved.")

In [ ]:
# Optional: upload the dataset and model artifact to S3.
import sagemaker

session = sagemaker.Session()
bucket = session.default_bucket()
prefix = "wine-quality-linear-regression"

data_s3_uri = session.upload_data(
    path="winequality-red.csv",
    bucket=bucket,
    key_prefix=f"{prefix}/input"
)
model_s3_uri = session.upload_data(
    path="linear_regression_no_custom_container.pkl",
    bucket=bucket,
    key_prefix=f"{prefix}/artifacts"
)

print(data_s3_uri)
print(model_s3_uri)

### Interpretation

This part does not build or manage a custom Docker image. Training occurs in the notebook environment with Scikit-learn. The saved artifact can be placed in S3 for deployment or grading evidence.

## Part B — Custom container in SageMaker

In [ ]:
import boto3
import sagemaker

region = boto3.Session().region_name
account_id = boto3.client("sts").get_caller_identity()["Account"]
role = sagemaker.get_execution_role()
ecr_repository = "wine-quality-sagemaker"
image_uri = (
    f"{account_id}.dkr.ecr.{region}.amazonaws.com/"
    f"{ecr_repository}:latest"
)

print("Region:", region)
print("Role:", role)
print("Image:", image_uri)

In [ ]:
ecr = boto3.client("ecr")

try:
    ecr.create_repository(repositoryName=ecr_repository)
    print("Created ECR repository.")
except ecr.exceptions.RepositoryAlreadyExistsException:
    print("ECR repository already exists.")

### Build and push the custom image

Run the displayed shell commands from an environment with Docker, AWS CLI access, and permission to push to ECR.

In [ ]:
login_command = (
    f"aws ecr get-login-password --region {region} | "
    f"docker login --username AWS --password-stdin "
    f"{account_id}.dkr.ecr.{region}.amazonaws.com"
)

print(login_command)
print(
    "docker build --platform linux/amd64 "
    "-t wine-quality-sagemaker ./sagemaker_container"
)
print(f"docker tag wine-quality-sagemaker:latest {image_uri}")
print(f"docker push {image_uri}")

In [ ]:
sm_session = sagemaker.Session()
bucket = sm_session.default_bucket()

train_s3_uri = sm_session.upload_data(
    path="winequality-red.csv",
    bucket=bucket,
    key_prefix="wine-quality-container/input"
)
print(train_s3_uri)

In [ ]:
from sagemaker.estimator import Estimator

estimator = Estimator(
    image_uri=image_uri,
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=f"s3://{bucket}/wine-quality-container/output",
    sagemaker_session=sm_session,
)

estimator.fit({"train": train_s3_uri})
print("Model artifact:", estimator.model_data)

## Cleanup

Delete any endpoint you create and stop unused Studio applications to avoid continuing charges. The training job stops after completion.

## Evidence to capture

Include screenshots of successful notebook output, the ECR image, the SageMaker training-job status, the model-artifact S3 URI, the GitHub notebook URL, and any CloudWatch logs used for troubleshooting.